In this file, we
1. automate the data preprocessing and
2. create multiplex horizontal visibility graphs for a certain time window in a file (edge list + adjacency matrix)

Comments:
- For the code to work, it needs to be in the same folder as the .edf and .seizure files.
- Window size can be adjusted; in my case the computer didn't have enough RAM for more than 5s.

What is missing:
- the code does not process files without a corresponding .seizure file
- actual graph creation for the correct windows (one wictal and one interictal)

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
#%pip install ts2vg
from ts2vg import HorizontalVG
import scipy.sparse as sp

In [5]:
# Extract seizure start and length from a .seizures annotation file.
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()
    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length

# Load an EDF file, create a DataFrame, and label seizure periods
def process_edf_with_labels(edf_file, seizure_file):
    raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)

    electrode_names = raw.ch_names
    sfreq           = raw.info['sfreq']
    n_samples       = len(raw.times)
    time_marks      = np.arange(n_samples) / sfreq
    electrode_data  = raw.get_data()

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = 'Time (s)'

    # label
    if seizure_file:
        seizure_start, seizure_length = get_seizure_period(seizure_file)
        seizure_end = seizure_start + seizure_length
        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
        has_seizure = True
    else:
        df["label"] = "interictal"
        has_seizure = False

    return df, has_seizure


# extract a window of exactly `window_size_s` seconds from the DataFrame
# if df cointains a seizure: window contains only ictal timepoints, if no seizure: random position
def extract_window(df, window_size_s=10):
    
    # infer number of samples per second from the time index
    sfreq = int(round(1 / (df.index[1] - df.index[0])))
    window_size_samples = window_size_s * sfreq

    has_seizure = (df["label"] == "ictal").any()

    if has_seizure:
        # ictal window
        ictal_indices = np.where(df["label"] == "ictal")[0]

        # find a contiguous ictal block large enough for the window
        # (handles cases where seizure is longer than 20s)
        valid_starts = [
            idx for idx in ictal_indices
            if idx + window_size_samples <= len(df)
            and (df["label"].iloc[idx : idx + window_size_samples] == "ictal").all()
        ]

        if not valid_starts:
            raise ValueError(
                f"No contiguous ictal block of {window_size_s}s found. "
                f"Seizure may be shorter than the window size."
            )

        # pick a random valid start within the ictal block
        start_idx = int(np.random.choice(valid_starts))

    else:
        # interictal window: random position anywhere in the file
        max_start = len(df) - window_size_samples
        if max_start <= 0:
            raise ValueError("DataFrame is shorter than the requested window size")
        start_idx = np.random.randint(0, max_start)

    window = df.iloc[start_idx : start_idx + window_size_samples]
    return window


# loop aover all .seizures files, process each edf and extract labelled windows
# returns: all_dataframes (dict of full labelled df's, windows (dict of extracted windows)
def process_all_files(pattern="*.seizures", window_size_s=10):
    seizure_files = sorted(glob.glob(pattern))

    if not seizure_files:
        print(f"No files found matching: {pattern}")
        return {}, {}

    all_dataframes = {}
    windows        = {}

    for i, seizure_file in enumerate(seizure_files):
        edf_file = seizure_file.replace(".seizures", "")

        try:
            # process full file
            df, has_seizure = process_edf_with_labels(edf_file, seizure_file)
            all_dataframes[edf_file] = df

            # extract window
            window = extract_window(df, window_size_s=window_size_s)

            # name window based on content
            window_name = f"window_ictal_{i}" if has_seizure else f"window_interictal_{i}"
            windows[window_name] = window

            # summary
            print(f"\n[{i}] {edf_file}")
            print(f"  Has seizure : {has_seizure}")
            print(f"  Window name : {window_name}")
            print(f"  Window size : {len(window)} samples ({window_size_s}s)")
            print(f"  Time range  : {window.index[0]:.1f}s → {window.index[-1]:.1f}s")
            print(f"  Labels      : {window['label'].value_counts().to_dict()}")

        except FileNotFoundError:
            print(f"ERROR: EDF file not found for {seizure_file} — expected {edf_file}")
        except ValueError as e:
            print(f"WARNING [{edf_file}]: {e}")
        except Exception as e:
            print(f"ERROR processing {seizure_file}: {e}")

    return all_dataframes, windows


# Run it
all_dataframes, windows = process_all_files("*.seizures", window_size_s=5)

# Access individual windows
# windows["window_ictal_0"]
# windows["window_interictal_2"]

# Check if all windows have the same number of samples
sizes = {name: len(df) for name, df in windows.items()}
print("\nWindow sizes:", sizes)

C:\Users\marin\AppData\Local\Temp\ipykernel_72884\818708623.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)



[0] chb01_03.edf
  Has seizure : True
  Window name : window_ictal_0
  Window size : 1280 samples (5s)
  Time range  : 3015.7s → 3020.7s
  Labels      : {'ictal': 1280}


C:\Users\marin\AppData\Local\Temp\ipykernel_72884\818708623.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)



[1] chb01_16.edf
  Has seizure : True
  Window name : window_ictal_1
  Window size : 1280 samples (5s)
  Time range  : 1028.3s → 1033.3s
  Labels      : {'ictal': 1280}


C:\Users\marin\AppData\Local\Temp\ipykernel_72884\818708623.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)



[2] chb07_12.edf
  Has seizure : True
  Window name : window_ictal_2
  Window size : 1280 samples (5s)
  Time range  : 1341.4s → 1346.4s
  Labels      : {'ictal': 1280}

Window sizes: {'window_ictal_0': 1280, 'window_ictal_1': 1280, 'window_ictal_2': 1280}


In [ ]:
# display(windows)
#display(windows["window_interictal_0"])

In [6]:
# build a multiplex horizontal visibility graph from an EEG window df
# -> each electrode is on one layer, each time point connects all electrodes (inter-layer edges)
# returns: edge list, adjacency matrix, node_index (dict for global node index), layer_info (dict with per-electrode HVG info)

def build_multiplex_hvg(window_df):
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes   = len(electrode_cols)
    n_timepoints   = len(window_df)
    n_nodes_total  = n_electrodes * n_timepoints

    print(f"Electrodes   : {n_electrodes}")
    print(f"Time points  : {n_timepoints}")
    print(f"Total nodes  : {n_nodes_total}")

    # 1. NODE INDEX: map (electrode_idx, time_idx) -> global node index
    # node layout: electrode 0 gets nodes 0..n_timepoints-1,
    #              electrode 1 gets nodes n_timepoints..2*n_timepoints-1, etc.

    def node_id(layer, t):
        return layer * n_timepoints + t

    node_index = {
        (electrode_cols[l], t): node_id(l, t)
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    }

    # 2. INTRA-LAYER EDGES: HVG edges within each electrode               
    intra_edges = []
    layer_info  = {}

    for l, electrode in enumerate(electrode_cols):
        ts  = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
            u = node_id(l, t_i)
            v = node_id(l, t_j)
            intra_edges.append({
                "source"       : u,
                "target"       : v,
                "source_label" : f"{electrode}_t{t_i}",
                "target_label" : f"{electrode}_t{t_j}",
                "edge_type"    : "intra",
                "layer"        : electrode,
            })

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index"  : l,
        }

    print(f"Intra-layer edges : {len(intra_edges)}")

    # 3. INTER-LAYER EDGES: connect same time point across all electrodes
    # at each time point t, connect every electrode to every other (full coupling)

    inter_edges = []

    for t in range(n_timepoints):
        for l_i in range(n_electrodes):
            for l_j in range(l_i + 1, n_electrodes):   # upper triangle only (undirected)
                u = node_id(l_i, t)
                v = node_id(l_j, t)
                inter_edges.append({
                    "source"       : u,
                    "target"       : v,
                    "source_label" : f"{electrode_cols[l_i]}_t{t}",
                    "target_label" : f"{electrode_cols[l_j]}_t{t}",
                    "edge_type"    : "inter",
                    "layer"        : f"{electrode_cols[l_i]}<->{electrode_cols[l_j]}",
                })

    print(f"Inter-layer edges : {len(inter_edges)}")

    # 4. COMBINED EDGE LIST 
    edge_list = pd.DataFrame(intra_edges + inter_edges)
    print(f"Total edges       : {len(edge_list)}")

    # 5. ADJACENCY MATRIX
    # use sparse matrix for efficiency (n_nodes can be large)
    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)

    for _, row in edge_list.iterrows():
        u, v = int(row["source"]), int(row["target"])
        adj_sparse[u, v] = 1
        adj_sparse[v, u] = 1  # undirected

    # convert to labeled DataFrame
    node_labels = [
        f"{electrode_cols[l]}_t{t}"
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    ]
    adj_matrix = pd.DataFrame(
        adj_sparse.toarray(),
        index   = node_labels,
        columns = node_labels
    )

    return edge_list, adj_matrix, node_index, layer_info


# Run on first ictal window 
ictal_key = next(k for k in windows if "ictal" in k)
window    = windows[ictal_key]

edge_list, adj_matrix, node_index, layer_info = build_multiplex_hvg(window)

# inspect results
print("\n--- Edge list (first 10) ---")
display(edge_list.head(10))

print("\n--- Adjacency matrix (first 5x5) ---")
display(adj_matrix.iloc[:5, :5])

print("\n--- Layer info ---")
display(pd.DataFrame(layer_info).T)

# split edge list back into intra / inter if needed
intra = edge_list[edge_list["edge_type"] == "intra"]
inter = edge_list[edge_list["edge_type"] == "inter"]

Electrodes   : 23
Time points  : 1280
Total nodes  : 29440
Intra-layer edges : 57041
Inter-layer edges : 323840
Total edges       : 380881

--- Edge list (first 10) ---


,source,target,source_label,target_label,edge_type,layer
0,725,726,FP1-F7_t725,FP1-F7_t726,intra,FP1-F7
1,726,727,FP1-F7_t726,FP1-F7_t727,intra,FP1-F7
2,724,725,FP1-F7_t724,FP1-F7_t725,intra,FP1-F7
3,471,725,FP1-F7_t471,FP1-F7_t725,intra,FP1-F7
4,727,728,FP1-F7_t727,FP1-F7_t728,intra,FP1-F7
5,727,729,FP1-F7_t727,FP1-F7_t729,intra,FP1-F7
6,727,730,FP1-F7_t727,FP1-F7_t730,intra,FP1-F7
7,727,733,FP1-F7_t727,FP1-F7_t733,intra,FP1-F7
8,470,471,FP1-F7_t470,FP1-F7_t471,intra,FP1-F7
9,149,471,FP1-F7_t149,FP1-F7_t471,intra,FP1-F7



--- Adjacency matrix (first 5x5) ---


,FP1-F7_t0,FP1-F7_t1,FP1-F7_t2,FP1-F7_t3,FP1-F7_t4
FP1-F7_t0,0,1,0,0,0
FP1-F7_t1,1,0,1,0,0
FP1-F7_t2,0,1,0,1,0
FP1-F7_t3,0,0,1,0,1
FP1-F7_t4,0,0,0,1,0



--- Layer info ---


,n_intra_edges,layer_index
FP1-F7,2506,0
F7-T7,2461,1
T7-P7,2458,2
P7-O1,2475,3
FP1-F3,2509,4
F3-C3,2469,5
C3-P3,2422,6
P3-O1,2456,7
FP2-F4,2533,8
F4-C4,2494,9
